# Level 3: Retrieval Quality (Hybrid Search)

This notebook compares Dense-Only retrieval (FAISS) against Hybrid Search (FAISS + BM25) across 3 different queries.

In [ ]:
import os
import json
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()

# Initialize models
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

with open('dataset.json', 'r', encoding='utf-8') as f:
    dataset = json.load(f)

docs = []
for item in dataset:
    doc = Document(page_content=item['content'], metadata={"title": item['title'], "url": item['url']})
    docs.append(doc)

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
all_chunks = text_splitter.split_documents(docs)

# Build Dense Retriever (FAISS)
vector_store = FAISS.from_documents(all_chunks, embeddings)
dense_retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# Build Sparse Retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(all_chunks)
sparse_retriever.k = 5

## Implement Reciprocal Rank Fusion (Hybrid)

In [ ]:
def hybrid_search(query, k=5):
    dense_results = dense_retriever.invoke(query)
    sparse_results = sparse_retriever.invoke(query)
    
    rrf_scores = {}
    for rank, doc in enumerate(dense_results):
        doc_content = doc.page_content
        if doc_content not in rrf_scores:
            rrf_scores[doc_content] = {"score": 0.0, "doc": doc}
        rrf_scores[doc_content]["score"] += 1.0 / (rank + 60)
            
    for rank, doc in enumerate(sparse_results):
        doc_content = doc.page_content
        if doc_content not in rrf_scores:
            rrf_scores[doc_content] = {"score": 0.0, "doc": doc}
        rrf_scores[doc_content]["score"] += 1.0 / (rank + 60)
        
    sorted_docs = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in sorted_docs][:k]

## Comparison: Dense vs Hybrid Search

In [ ]:
test_queries = [
    "What is Azure Chaos Studio?", # Niche/Specific terminology
    "How do I manage relational databases?", # Broad semantic query
    "Error 404 in Azure Web Apps" # Keyword-heavy query
]

for q in test_queries:
    print(f"\n=======================================================")
    print(f"QUERY: {q}")
    print(f"=======================================================\n")
    
    print("--- TOP 3 DENSE RESULTS (FAISS) ---")
    dense_docs = dense_retriever.invoke(q)[:3]
    for i, doc in enumerate(dense_docs):
        print(f"{i+1}. [{doc.metadata['title']}] {doc.page_content[:100]}...")
        
    print("\n--- TOP 3 HYBRID RESULTS (FAISS + BM25) ---")
    hybrid_docs = hybrid_search(q, k=3)
    for i, doc in enumerate(hybrid_docs):
        print(f"{i+1}. [{doc.metadata['title']}] {doc.page_content[:100]}...")
        
    print("\n\n")